# 67 · Bootstrap Oλ survival sensitivity
For each 10,000-cell replicate, convert the historical `O_lambda` column (Tλ) to Oλ = 1/Tλ. Each patient contributes one survival observation with 25 equally weighted predictor particles. The pooled synthetic analysis uses the target patient's outcome and clinical covariates. Run all cells in Colab. Completed fits are reused when inputs and sampler settings match.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import pandas as pd

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
REFERENCE = DATA / 'olambda_reciprocal_direct_no_asymptote_floor_v1' / 'patient_olambda_asymptotes.csv'
OUTCOMES = DATA / 'clinical_outcomes.csv'
OUTPUT_ROOT = DATA / 'bayesian_Olambda_sensitivity_survival' / 'direct_reciprocal_adjusted_midpoint_v1'
SOURCES = {'patient-bootstrap': 'patient_bayesian_bootstrap',
           'pooled-synthetic': 'atlas_hard_label_pooled_synthetic'}
# Optional explicit paths to the original replicate CSVs.
SOURCE_CSV = {}
DRAWS, TUNE, CHAINS, CORES = 2000, 2000, 4, 4
TARGET_ACCEPT, SEED = 0.99, 0
import numpy as np
from IPython.display import display

In [ ]:
REVISION = 'ad6cbc85518f75aa2076dfbb18d9a9da9cc104ec'
repo = Path(tempfile.mkdtemp(prefix='direct-survival-')) / 'repo'
subprocess.run(['git', 'clone', '--quiet', '--single-branch', '--branch', 'direct-olambda',
                'https://github.com/adamnik/CAR-T-spectral-organization.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--quiet', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymc==5.28.5',
                'arviz>=0.22,<1', 'h5netcdf>=1.5'], check=True)
print('Survival model revision:', REVISION)

In [ ]:
keys = ['Dataset', 'sample_id']
reference = pd.read_csv(REFERENCE).set_index(keys, verify_integrity=True)
assert len(reference) == 163 and reference.index.get_level_values('Dataset').nunique() == 8
replicate_ids = list(range(25))
def prepare(experiment, folder):
    source = DATA/'olambda_sensitivity'/folder
    if experiment in SOURCE_CSV:
        paths = [Path(SOURCE_CSV[experiment])]
    elif (source/'olambda_sensitivity_replicates.csv').is_file():
        paths = [source/'olambda_sensitivity_replicates.csv']
    else:
        paths = sorted((source/'replicate_parts').glob('*.csv'))
    if not paths:
        raise FileNotFoundError(f'No replicates in {source}. Set SOURCE_CSV[{experiment!r}] to the saved replicate CSV.')
    cols = [*keys, 'experiment', 'replicate', 'O_lambda', 'n_cells_native', 'n_cells_target']
    d = pd.concat([pd.read_csv(path, usecols=cols) for path in paths], ignore_index=True)
    assert d.experiment.eq(experiment).all() and d.n_cells_target.eq(10000).all()
    assert d[keys].notna().all().all() and d.n_cells_native.ge(1000).all()
    d = d.rename(columns={'O_lambda':'T_lambda'})
    assert np.isfinite(d.T_lambda).all() and d.T_lambda.gt(1).all(), 'Check historical lifetime scale before inverting.'
    d['O_lambda'] = 1.0 / d.T_lambda
    particles = d.pivot(index=keys, columns='replicate', values='O_lambda').sort_index()
    assert list(particles) == replicate_ids and particles.notna().all().all()
    missing = reference.index.difference(particles.index)
    assert particles.index.difference(reference.index).empty
    allowed_missing = {('Maurer','CAR137_maurer')} if experiment == 'pooled-synthetic' else set()
    assert set(missing) <= allowed_missing, f'Unexpected missing patients: {list(missing)}'
    groups = d.groupby(keys)
    assert groups.n_cells_native.nunique().eq(1).all()
    estimates = pd.DataFrame({'n_cells_available':groups.n_cells_native.first(),
                              'O_lambda_inf_central':particles.mean(axis=1),
                              'olambda_scale':'reciprocal'})
    dest = OUTPUT_ROOT/folder/'prepared_inputs'
    dest.mkdir(parents=True, exist_ok=True)
    ep, pp = dest/'patient_estimates.csv', dest/'patient_particles.csv'
    estimates.to_csv(ep)
    particles.columns = [f'particle_{i:02d}' for i in range(1,26)]
    particles.to_csv(pp)
    d.sort_values([*keys,'replicate']).to_csv(dest/'reciprocal_replicates.csv', index=False)
    missing.to_frame(index=False).to_csv(dest/'absent_from_replicates.csv',index=False)
    print(experiment, '| patients:', len(estimates), '| absent from replicates:', list(missing), flush=True)
    return ep, pp

inputs = {experiment:prepare(experiment, folder) for experiment, folder in SOURCES.items()}

In [ ]:
audit = """
from survival.survival_model import load_olambda_analysis
import sys, pandas as pd
ep, pp, outcomes = sys.argv[1:]
data, *_ = load_olambda_analysis(ep, outcomes, 1000, 25, pp, ['Treatment','Disease'])
keys = ['Dataset','sample_id']
expected = pd.read_csv(ep).set_index(keys).index.sort_values()
assert expected.equals(data.set_index(keys).index.sort_values()), 'Clinical join changed the replicate cohort.'
print(data.groupby('Dataset').agg(patients=('sample_id','size'),events=('event_observed','sum')).to_string())
"""
for experiment,(ep, pp) in inputs.items():
    print(experiment, flush=True)
    subprocess.run([sys.executable,'-c',audit,ep,pp,OUTCOMES], cwd=repo,check=True)

In [ ]:
def fit(experiment, module, name, extra=()):
    ep, pp = inputs[experiment]
    common = ['--particles-csv', str(pp), '--minimum-cells', '1000', '--n-particles', '25',
              '--draws', str(DRAWS), '--tune', str(TUNE), '--chains', str(CHAINS), '--cores', str(CORES),
              '--target-accept', str(TARGET_ACCEPT), '--random-state', str(SEED),
              '--adjustment', 'Treatment=axicel', '--adjustment', 'Disease=lymphoma']
    hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (ep, pp, OUTCOMES)}

    config = dict(revision=REVISION, module=module, arguments=common+list(extra), input_sha256=hashes)
    tag = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
    dest = OUTPUT_ROOT / SOURCES[experiment] / name / tag
    marker = dest / 'run_manifest.json'  # Written by the model after the posterior and all tables.
    if marker.is_file():
        print('Using completed fit:', dest, flush=True)
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        stage = Path(tempfile.mkdtemp(prefix=tag+'-running-', dir=dest.parent))
        print('Fitting', name, '| output:', dest, flush=True)
        command = [sys.executable, '-u', '-m', 'survival.'+module, ep, OUTCOMES,
                   stage, *common, *extra]
        with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
            if proc.wait():
                raise subprocess.CalledProcessError(proc.returncode, command)
        (stage / 'notebook_config.json').write_text(json.dumps(config, indent=2))
        stage.rename(dest)
    if json.loads((dest/'notebook_config.json').read_text()) != config:
        raise ValueError(f'Run configuration mismatch: {dest}')
    print('Saved posterior:', dest/'posterior.nc')
    display(pd.read_csv(dest/'model_result.csv'))
    return dest

In [ ]:
rows = []
for experiment in SOURCES:
    for name, module, extra in [
        ('continuous','fit_olambda_survival_continuous',[]),
        ('smooth_transition_Q10_Q90_midpoint','fit_olambda_survival_regimes',
         ['--midpoint-centered','--crossover-quantile-low','0.10',
          '--crossover-quantile-high','0.90','--transition-scale-sd','0.10']),
    ]:
        print(experiment, name, flush=True)
        dest = fit(experiment,module,name,extra)
        if extra:
            manifest = json.loads((dest/'run_manifest.json').read_text())
            assert manifest['hazard_activation_offset'] == 0.5
        result = pd.read_csv(dest/'model_result.csv')
        rows.append(result.assign(experiment=experiment, fit=name, output_dir=str(dest)))
        combined = pd.concat(rows, ignore_index=True)
        combined.to_csv(OUTPUT_ROOT/'sensitivity_survival_model_comparison.csv',index=False)
display(combined[['experiment','fit','patients','HR_median','HR_q025','HR_q975',
                  'P_all_datasets_same_direction','divergences','max_rhat']])
print('Saved:', OUTPUT_ROOT)